# Historical LSTM Silver-Label Generation

This notebook documents the original LSTM labelling workflow: raw-review eligibility screening, SmSA preparation, bidirectional LSTM training, batch inference, and persistence of the silver-labelled dataset and checkpoint artifacts.


## Release Boundary

This is a restricted code-inspection copy. It retains the original analytical logic, but it does not include Banjarbaru source data, review-level records, manual-audit files, trained checkpoints, or intermediate outputs. The paths in the code refer to the authorized analysis environment and are not expected to run from this public package.


### Install required packages

Installs the PyTorch, TorchText, and Hugging Face Datasets dependencies used by the original training environment.


In [ ]:
!pip install torch torchtext datasets -q
print("Library berhasil diinstall!")


### Import libraries and select the compute device

Imports the data, PyTorch, plotting, and progress utilities. It also fixes the historical random seed and selects CUDA when it was available in the authorized environment.


In [ ]:
import pandas as pd
import numpy as np
import re
import os
from collections import Counter
import warnings
import pickle
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Library berhasil diimport!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")


### Load the authorized raw review workbook

Loads the restricted workbook and reports its row count and available columns. The workbook is not distributed.


In [ ]:
DATA_PATH = '../data/raw/#######.xlsx'

df_raw = pd.read_excel(DATA_PATH)

print("="*60)
print("STATISTIK DATA AWAL (SEBELUM SELEKSI)")
print("="*60)
print(f"Total entri awal: {len(df_raw):,} baris")
print(f"\nKolom yang tersedia:")
for col in df_raw.columns:
    print(f"  - {col}")


### Inspect the input review field

Displays a small local preview of place names and raw review text to verify that the expected review column was loaded. No preview output is released.


In [ ]:
print("\nPreview 10 baris pertama (kolom text_raw):")
df_raw[['Place', 'text_raw']].head(10)


### Quantify review availability before cleaning

Counts placeholder strings and null values in text_raw to establish how many source records contain usable review text.


In [ ]:
print("\n" + "="*60)
print("ANALISIS KOLOM 'text_raw' SEBELUM PEMBERSIHAN")
print("="*60)

total = len(df_raw)
tidak_ada_review = df_raw['text_raw'].str.contains('Tidak ada review', case=False, na=False).sum()
ulasan_tidak_ditemukan = df_raw['text_raw'].str.contains('Ulasan tidak ditemukan', case=False, na=False).sum()
null_values = df_raw['text_raw'].isna().sum()
ada_ulasan = total - tidak_ada_review - ulasan_tidak_ditemukan - null_values

print(f"\nDistribusi ulasan:")
print(f"  - Total entri: {total:,}")
print(f"  - 'Tidak ada review': {tidak_ada_review:,} ({tidak_ada_review/total*100:.2f}%)")
print(f"  - 'Ulasan tidak ditemukan': {ulasan_tidak_ditemukan:,} ({ulasan_tidak_ditemukan/total*100:.2f}%)")
print(f"  - Nilai null/NaN: {null_values:,} ({null_values/total*100:.2f}%)")
print(f"  - Memiliki ulasan teks: {ada_ulasan:,} ({ada_ulasan/total*100:.2f}%)")


### Define eligibility helpers

Defines the emoji-only detector and word-count function used to exclude non-linguistic and very short review records.


In [ ]:
def is_emoji_only(text):

    if pd.isna(text):
        return False
    
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002702-\U000027B0"  # dingbats
        "\U0001F900-\U0001F9FF"  # supplemental symbols
        "\U0001FA00-\U0001FA6F"  # chess symbols
        "\U0001FA70-\U0001FAFF"  # symbols and pictographs extended-a
        "\U00002600-\U000026FF"  # misc symbols
        "]+", 
        flags=re.UNICODE
    )
    
    text_without_emoji = emoji_pattern.sub('', str(text))
    text_clean = re.sub(r'[^\w\s]', '', text_without_emoji).strip()
    
    return len(text_clean) == 0


def count_words(text):
 
    if pd.isna(text):
        return 0
    
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002600-\U000026FF"
        "]+", 
        flags=re.UNICODE
    )
    text_clean = emoji_pattern.sub(' ', str(text))
    
    words = [w for w in text_clean.split() if re.search(r'\w', w)]
    return len(words)

test_texts = [
    "Bagus sekali tempatnya",
    "\U0001F44D\U0001F44D\U0001F44D",
    "Mantap \U0001F44D",
    "ok",
    "\U0001F60A"
]

print("Test fungsi pengecekan:")
for t in test_texts:
    print(f"  '{t}' -> emoji_only={is_emoji_only(t)}, word_count={count_words(t)}")


### Apply the historical raw-review eligibility rules

Removes placeholder, missing, emoji-only, very short, and duplicate review texts, then retains the cleaned records for LSTM labelling.


In [ ]:
df = df_raw.copy()

print("="*60)
print("PROSES PEMBERSIHAN DATA")
print("="*60)

before = len(df)
df = df[~df['text_raw'].str.contains('Tidak ada review', case=False, na=False)]
print(f"\n[1] Hapus 'Tidak ada review': {before:,} -> {len(df):,} (dihapus: {before - len(df):,})")

before = len(df)
df = df[~df['text_raw'].str.contains('Ulasan tidak ditemukan', case=False, na=False)]
print(f"[2] Hapus 'Ulasan tidak ditemukan': {before:,} -> {len(df):,} (dihapus: {before - len(df):,})")

before = len(df)
df = df.dropna(subset=['text_raw'])
print(f"[3] Hapus nilai null/NaN: {before:,} -> {len(df):,} (dihapus: {before - len(df):,})")

before = len(df)
df['is_emoji_only'] = df['text_raw'].apply(is_emoji_only)
emoji_only_samples = df[df['is_emoji_only']][['text_raw']].head(10)
df = df[~df['is_emoji_only']]
df = df.drop(columns=['is_emoji_only'])
print(f"[4] Hapus ulasan hanya emoji: {before:,} -> {len(df):,} (dihapus: {before - len(df):,})")

before = len(df)
df['word_count'] = df['text_raw'].apply(count_words)
short_review_samples = df[df['word_count'] < 3][['text_raw', 'word_count']].head(20)
df = df[df['word_count'] >= 3]
print(f"[5] Hapus ulasan < 3 kata: {before:,} -> {len(df):,} (dihapus: {before - len(df):,})")

before = len(df)
df = df.drop_duplicates(subset=['text_raw'], keep='first')
print(f"[6] Hapus duplikat: {before:,} -> {len(df):,} (dihapus: {before - len(df):,})")

print(f"\n" + "="*60)
print(f"HASIL AKHIR: {len(df):,} baris dari {len(df_raw):,} baris awal")
print(f"Persentase data yang dipertahankan: {len(df)/len(df_raw)*100:.2f}%")
print("="*60)


### Inspect excluded examples locally

Displays local examples of emoji-only and short reviews as a quality-control check. These review-level examples are not included in the release.


In [ ]:
print("Contoh ulasan yang hanya berisi emoji (dihapus):")
display(emoji_only_samples)

print("\nContoh ulasan dengan < 3 kata (dihapus):")
display(short_review_samples)


### Summarize retained review lengths

Calculates word and character length summaries after the eligibility rules have been applied.


In [ ]:
print("="*60)
print("STATISTIK DATA SETELAH PEMBERSIHAN")
print("="*60)

print(f"\nStatistik panjang ulasan (jumlah kata):")
print(f"  - Minimum: {df['word_count'].min()} kata")
print(f"  - Maximum: {df['word_count'].max()} kata")
print(f"  - Rata-rata: {df['word_count'].mean():.2f} kata")
print(f"  - Median: {df['word_count'].median():.0f} kata")
print(f"  - Std Dev: {df['word_count'].std():.2f}")

df['char_count'] = df['text_raw'].apply(lambda x: len(str(x)))
print(f"\nStatistik panjang ulasan (jumlah karakter):")
print(f"  - Minimum: {df['char_count'].min()} karakter")
print(f"  - Maximum: {df['char_count'].max()} karakter")
print(f"  - Rata-rata: {df['char_count'].mean():.2f} karakter")
print(f"  - Median: {df['char_count'].median():.0f} karakter")


### Retrieve the SmSA reference corpus

Downloads the SmSA split from IndoNLU, which supplied the source-domain sentiment supervision for the LSTM labeler.


In [ ]:
from datasets import load_dataset

print("Downloading SmSA dataset dari IndoNLU...")

smsa_dataset = load_dataset(
    'indonlp/indonlu',
    'smsa',
    trust_remote_code=True
)

print(f"\nDataset SmSA:")
print(f"  - Train: {len(smsa_dataset['train']):,} samples")
print(f"  - Validation: {len(smsa_dataset['validation']):,} samples")
print(f"  - Test: {len(smsa_dataset['test']):,} samples")

print(f"\nContoh data:")
for i in range(3):
    print(f"  [{smsa_dataset['train'][i]['label']}] {smsa_dataset['train'][i]['text'][:80]}...")


### Materialize SmSA splits and class mapping

Converts the SmSA train, validation, and test partitions to data frames and defines the three-class label mapping used by the LSTM.


In [ ]:
train_df = pd.DataFrame(smsa_dataset['train'])
val_df = pd.DataFrame(smsa_dataset['validation'])
test_df = pd.DataFrame(smsa_dataset['test'])

label_map = {0: 'positif', 1: 'netral', 2: 'negatif'}
label_map_reverse = {'positif': 0, 'netral': 1, 'negatif': 2}

print("Distribusi label pada training data:")
print(train_df['label'].value_counts().sort_index())
print(f"\nLabel mapping: {label_map}")


### Define SmSA text normalization and tokenization

Lowercases SmSA text, removes URLs, mentions, hashtags, and punctuation, then tokenizes it on whitespace for vocabulary construction.


In [ ]:
def preprocess_text(text):

    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    text = re.sub(r'@\w+|#\w+', '', text)
    
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def tokenize(text):

    return text.split()


test_text = "Makanannya  enak banget!!! @user #recommended https://example.com"
print(f"Original: {test_text}")
print(f"Preprocessed: {preprocess_text(test_text)}")
print(f"Tokens: {tokenize(preprocess_text(test_text))}")


### Build the SmSA vocabulary

Builds a training-corpus vocabulary with PAD and UNK tokens and a minimum token-frequency threshold of two.


In [ ]:
class Vocabulary:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.word_freq = Counter()
        self.min_freq = min_freq
        
    def build_vocab(self, texts):
        for text in texts:
            tokens = tokenize(preprocess_text(text))
            self.word_freq.update(tokens)
        
        idx = len(self.word2idx)
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq and word not in self.word2idx:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1
                
    def encode(self, text):
        tokens = tokenize(preprocess_text(text))
        return [self.word2idx.get(token, self.word2idx['<UNK>']) for token in tokens]
    
    def __len__(self):
        return len(self.word2idx)


print("Building vocabulary...")
vocab = Vocabulary(min_freq=2)
vocab.build_vocab(train_df['text'].tolist())

print(f"Vocabulary size: {len(vocab):,}")
print(f"\nContoh encoding:")
test_text = "makanan enak sekali"
print(f"  '{test_text}' -> {vocab.encode(test_text)}")


### Prepare PyTorch datasets and loaders

Encodes SmSA train and validation texts, truncates sequences to 128 tokens, pads each batch, and creates attention masks.


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoded = self.vocab.encode(text)
        
        if len(encoded) > self.max_len:
            encoded = encoded[:self.max_len]
        
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(label, dtype=torch.long)


def collate_fn(batch):

    texts, labels = zip(*batch)
    
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    
    attention_mask = (texts_padded != 0).float()
    
    return texts_padded, labels, attention_mask


MAX_LEN = 128
BATCH_SIZE = 64

train_dataset = SentimentDataset(
    train_df['text'].tolist(),
    train_df['label'].tolist(),
    vocab,
    MAX_LEN
)

val_dataset = SentimentDataset(
    val_df['text'].tolist(),
    val_df['label'].tolist(),
    vocab,
    MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


### Define the bidirectional LSTM classifier

Constructs the three-class bidirectional LSTM with 128-dimensional embeddings, two recurrent layers, dropout, masked mean pooling, and dense output layers.


In [ ]:
class LSTMSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, 
                 bidirectional=True, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        
        self.fc_hidden = hidden_dim * 2 if bidirectional else hidden_dim
        
        self.fc1 = nn.Linear(self.fc_hidden, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        
    def forward(self, text, attention_mask=None):
        embedded = self.embedding(text)  # [batch_size, seq_len, embedding_dim]
        embedded = self.dropout(embedded)
        
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1)  # [batch_size, seq_len, 1]
            lstm_out_masked = lstm_out * mask
            sum_output = lstm_out_masked.sum(dim=1)  # [batch_size, hidden_dim * 2]
            lengths = mask.sum(dim=1).clamp(min=1)  # Avoid division by zero
            pooled = sum_output / lengths
        else:
            pooled = lstm_out.mean(dim=1)
        
        fc1_out = self.relu(self.fc1(self.dropout(pooled)))
        output = self.fc2(self.dropout(fc1_out))
        
        return output


VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
OUTPUT_DIM = 3  # 3 classes: negative, neutral, positive
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.3

model = LSTMSentimentClassifier(
    VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, 
    N_LAYERS, BIDIRECTIONAL, DROPOUT
).to(device)

print(f"Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


### Configure optimisation and evaluation functions

Sets cross-entropy loss, Adam optimisation, ReduceLROnPlateau scheduling, gradient clipping, and epoch-level training and validation routines.


In [ ]:
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)


def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for texts, labels, attention_mask in tqdm(dataloader, desc="Training", leave=False):
        texts = texts.to(device)
        labels = labels.to(device)
        attention_mask = attention_mask.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(texts, attention_mask)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for texts, labels, attention_mask in dataloader:
            texts = texts.to(device)
            labels = labels.to(device)
            attention_mask = attention_mask.to(device)
            
            outputs = model(texts, attention_mask)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(dataloader), correct / total


print("Training setup complete!")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Epochs: {NUM_EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE}")


### Train and retain the lowest-validation-loss checkpoint

Runs up to ten epochs on SmSA, records train and validation loss and accuracy, and restores the parameter state with the lowest validation loss.


In [ ]:
print("="*60)
print("TRAINING LSTM MODEL")
print("="*60)

best_val_loss = float('inf')
best_model_state = None
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        print(f"  * New best model saved!")
    print()

model.load_state_dict(best_model_state)
print(f"Best validation loss: {best_val_loss:.4f}")


### Perform a local inference smoke test

Applies the trained labeler to a few hand-written Indonesian review examples to verify the prediction interface and probability order.


In [ ]:
def predict_sentiment(model, text, vocab, device):

    model.eval()
    
    encoded = vocab.encode(text)
    if len(encoded) == 0:
        encoded = [vocab.word2idx['<UNK>']]
    
    text_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
    attention_mask = torch.ones_like(text_tensor, dtype=torch.float).to(device)
    
    with torch.no_grad():
        outputs = model(text_tensor, attention_mask)
        probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
    
    label_idx = predicted.item()
    label = label_map[label_idx]
    probs = probabilities[0].cpu().numpy()
    
    return label, probs


test_reviews = [
    "Makanannya enak banget dan pelayanan ramah",
    "Tempatnya kotor dan pelayanan buruk sekali",
    "Biasa saja tidak ada yang spesial",
    "Makanannya enak tapi pelayanannya lambat",
    "Sangat recommended! Harga terjangkau kualitas mantap",
    "Tidak akan kesini lagi, mengecewakan"
]

print("="*60)
print("TEST PREDIKSI SENTIMEN LSTM")
print("="*60)

for review in test_reviews:
    label, probs = predict_sentiment(model, review, vocab, device)
    print(f"\nUlasan: '{review}'")
    print(f"  Prediksi: {label.upper()}")
    print(f"  Probabilitas: Positif={probs[0]:.3f}, Netral={probs[1]:.3f}, Negatif={probs[2]:.3f}")


### Generate LSTM-derived silver labels for eligible reviews

Runs batch inference over the authorized cleaned reviews and appends the predicted class probabilities, maximum probability, and silver sentiment label.


In [ ]:
def batch_predict(model, texts, vocab, device, batch_size=128):

    model.eval()
    all_labels = []
    all_probs = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
        batch_texts = texts[i:i+batch_size]
        
        encoded_batch = []
        for text in batch_texts:
            encoded = vocab.encode(text)
            if len(encoded) == 0:
                encoded = [vocab.word2idx['<UNK>']]
            encoded_batch.append(torch.tensor(encoded, dtype=torch.long))
        
        texts_padded = pad_sequence(encoded_batch, batch_first=True, padding_value=0).to(device)
        attention_mask = (texts_padded != 0).float().to(device)
        
        with torch.no_grad():
            outputs = model(texts_padded, attention_mask)
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
        
        all_labels.extend(predicted.cpu().numpy())
        all_probs.extend(probabilities.cpu().numpy())
    
    return all_labels, all_probs


print("Melakukan pelabelan sentimen LSTM pada dataset Google Maps...")
print(f"Total data: {len(df):,} baris\n")

labels, probs = batch_predict(model, df['text_raw'].tolist(), vocab, device)

df['sentiment_label'] = [label_map[l] for l in labels]
df['prob_positif'] = [p[0] for p in probs]
df['prob_netral'] = [p[1] for p in probs]
df['prob_negatif'] = [p[2] for p in probs]
df['confidence'] = [max(p) for p in probs]

print("\nPelabelan selesai!")


### Summarize the silver-label class distribution

Reports the count, percentage, and imbalance ratio for the three LSTM-derived classes.


In [ ]:
print("="*60)
print("DISTRIBUSI LABEL SENTIMEN (LSTM)")
print("="*60)

label_counts = df['sentiment_label'].value_counts()
label_percentages = df['sentiment_label'].value_counts(normalize=True) * 100

print(f"\nDistribusi kelas:")
for label in ['positif', 'netral', 'negatif']:
    if label in label_counts.index:
        print(f"  {label.capitalize()}: {label_counts[label]:,} ({label_percentages[label]:.2f}%)")

max_class = label_counts.max()
min_class = label_counts.min()
imbalance_ratio = max_class / min_class
print(f"\nRasio imbalance (max/min): {imbalance_ratio:.2f}:1")
if imbalance_ratio > 3:
    print("WARNING: Class imbalance > 3:1, pertimbangkan strategi penyeimbangan!")


### Create the historical silver-label distribution figure

Builds a local bar-and-pie summary of the silver-label distribution. The figure is a historical diagnostic and is not released as a current manuscript result.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'positif': '#2ecc71', 'netral': '#3498db', 'negatif': '#e74c3c'}
label_order = ['positif', 'netral', 'negatif']
counts = [label_counts.get(l, 0) for l in label_order]
bars = axes[0].bar(label_order, counts, color=[colors[l] for l in label_order], edgecolor='black')
axes[0].set_xlabel('Sentimen')
axes[0].set_ylabel('Jumlah')
axes[0].set_title('Distribusi Label Sentimen (LSTM)')
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')

axes[1].pie(counts, labels=[f"{l.capitalize()}\n({c:,})" for l, c in zip(label_order, counts)],
            colors=[colors[l] for l in label_order], autopct='%1.1f%%',
            explode=(0.02, 0.02, 0.02), shadow=True)
axes[1].set_title('Proporsi Label Sentimen (LSTM)')

plt.tight_layout()
plt.savefig('../figures/(2)distribusi_sentimen_lstm.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nGrafik disimpan ke: figures/(2)distribusi_sentimen_lstm.png")


### Inspect local examples by silver label

Displays local review examples and their maximum probabilities within each predicted sentiment class. No review text is released.


In [ ]:
print("="*60)
print("CONTOH HASIL PELABELAN (LSTM)")
print("="*60)

for label in ['positif', 'negatif', 'netral']:
    print(f"\n--- Contoh ulasan {label.upper()} ---")
    samples = df[df['sentiment_label'] == label][['text_raw', 'confidence']].head(5)
    for idx, row in samples.iterrows():
        text_display = row['text_raw'][:100] + "..." if len(row['text_raw']) > 100 else row['text_raw']
        print(f"  [conf={row['confidence']:.3f}] {text_display}")


### Summarize LSTM confidence values

Reports the overall and class-specific maximum-probability distribution, including the number of low-confidence LSTM predictions.


In [ ]:
print("="*60)
print("ANALISIS CONFIDENCE SCORE")
print("="*60)

print(f"\nStatistik confidence:")
print(f"  - Minimum: {df['confidence'].min():.4f}")
print(f"  - Maximum: {df['confidence'].max():.4f}")
print(f"  - Rata-rata: {df['confidence'].mean():.4f}")
print(f"  - Median: {df['confidence'].median():.4f}")

print(f"\nRata-rata confidence per kelas:")
for label in ['positif', 'netral', 'negatif']:
    mean_conf = df[df['sentiment_label'] == label]['confidence'].mean()
    print(f"  {label.capitalize()}: {mean_conf:.4f}")

low_conf_threshold = 0.5
low_conf_count = (df['confidence'] < low_conf_threshold).sum()
print(f"\nSampel dengan confidence < {low_conf_threshold}: {low_conf_count:,} ({low_conf_count/len(df)*100:.2f}%)")


### Select and persist the silver-labelled dataset

Keeps the place identifiers, review text, review-length fields, three class probabilities, confidence, and LSTM-derived sentiment label, then writes those restricted fields to the historical labelled-review CSV. The row-level file is not included in.


In [ ]:
columns_to_save = [
    'Place', 'Actual Place Name', 'Place Type',
    'text_raw', 'word_count', 'char_count',
    'prob_positif', 'prob_netral', 'prob_negatif', 'confidence',
    'sentiment_label'
]

columns_to_save = [c for c in columns_to_save if c in df.columns]
df_labeled = df[columns_to_save].copy()

df_labeled = df_labeled.reset_index(drop=True)

output_path = '../data/interim/labeled_reviews_lstm.csv'
df_labeled.to_csv(output_path, index=False, encoding='utf-8')

print(f"Dataset tersimpan ke: {output_path}")
print(f"Total baris: {len(df_labeled):,}")
print(f"\nKolom yang disimpan:")
for col in df_labeled.columns:
    print(f"  - {col}")


### Persist the LSTM checkpoint and vocabulary

Writes the trained model state, architecture metadata, label map, and vocabulary used by the LSTM provenance check. These artifacts are not included.


In [ ]:
import pickle

MODEL_DIR = '../models/lstm_labeler'
os.makedirs(MODEL_DIR, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': VOCAB_SIZE,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': HIDDEN_DIM,
    'output_dim': OUTPUT_DIM,
    'n_layers': N_LAYERS,
    'bidirectional': BIDIRECTIONAL,
    'dropout': DROPOUT,
    'label_map': label_map,
}, os.path.join(MODEL_DIR, 'lstm_sentiment_model.pt'))

with open(os.path.join(MODEL_DIR, 'vocabulary.pkl'), 'wb') as f:
    pickle.dump(vocab, f)

print(f"Model dan vocabulary tersimpan ke: {MODEL_DIR}/")
print(f"  - lstm_sentiment_model.pt")
print(f"  - vocabulary.pkl")
